# Land climate assessment — 1986–2005 D03 nest (173-station canonical set)

Longest-available climatology of the D03 land observation network and of each model, per station and pooled. This is the *climate* of the evaluation set: period-mean (climatology) values and how well each model reproduces them, supplementing the anomaly/bias tables in the report.

**Data source (no re-parse of raw files):** `pickles/taylor_annuals.pkl` (v10) carries the validated (station × year) annual values — `ta[var] = (obs_dict, models_dict)` — already restricted to each obs series' surviving years (Rule-1 90% annual gate, like-for-like). The canonical D03 finite land set (T 146 / P 159 / W 16) is taken from `station_tables.pkl`. Each climatology is the mean over the station's surviving years (20 yr if all pass, fewer for sparse records — `n_years` reported per site).

In [11]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import matplotlib.cm as cm
import matplotlib.colorbar
BASE = os.getcwd()
with open(os.path.join(BASE,'pickles','taylor_annuals.pkl'),'rb') as f: TA = pickle.load(f)
with open(os.path.join(BASE,'pickles','station_tables.pkl'),'rb') as f: ST = pickle.load(f)
D03 = ST['D03'].copy()
D03['sid'] = D03['station'].astype(str)
print('taylor version:', TA['_version'])
print('station table rows:', len(D03))
VARS = {'t':'T (°C)','pr':'P (mm/yr)','wind':'W (m/s)'}
MODELS = ['D03','D02','D01','CanESM2','CanRCM4']
MNAME = {'D03':'D3','D02':'D15','D01':'D75','CanESM2':'CanESM2','CanRCM4':'CanRCM4'}
# canonical per-var land sets from the table's finite bias columns
CANON = {'t': D03[D03['temperature_bias_C'].notna()]['sid'].tolist(),
         'pr': D03[D03['precipitation_bias_pct'].notna()]['sid'].tolist(),
         'wind': D03[D03['wind_bias_pct'].notna()]['sid'].tolist()}
for v in VARS: print(f'{v}: canon n = {len(CANON[v])}')

taylor version: 10
station table rows: 173
t: canon n = 146
pr: canon n = 159
wind: canon n = 16


In [12]:
def climo(obs, mods, sids):
    """Per-site period climatology (mean over surviving years) + pooled means.
    Like-for-like: each model is masked to the obs surviving years for that site."""
    rows=[]
    for s in sids:
        if s not in obs: continue
        o = np.asarray(obs[s], float)
        ok = np.isfinite(o)
        rec = {'sid': s, 'n_years': int(ok.sum()),
               'obs': o[ok].mean() if ok.any() else np.nan}
        for m in MODELS:
            if s in mods[m]:
                mraw = np.asarray(mods[m][s], float)
                shared = ok & np.isfinite(mraw)          # obs surviving AND model valid
                rec['obs_'+m] = o[shared].mean() if shared.any() else np.nan
                mv = mraw[ok]                            # model over obs surviving years
                rec[m] = mv[np.isfinite(mv)].mean() if np.isfinite(mv).any() else np.nan
            else:
                rec['obs_'+m] = np.nan; rec[m] = np.nan
        rows.append(rec)
    df = pd.DataFrame(rows).set_index('sid')
    pooled = {'obs': df['obs'].mean()}
    for m in MODELS:
        mm = df['obs_'+m].notna() & df[m].notna()
        pooled[m] = df.loc[mm, m].mean() if mm.any() else np.nan
    return df, pooled

In [13]:
# ---- per-variable site climatologies ----
SITES={}; POOLED={}
for v in ['t','pr','wind']:
    SITES[v], POOLED[v] = climo(TA[v][0], TA[v][1], CANON[v])
    print(f'{v:5s}: {len(SITES[v]):3d} sites, obs pooled = {POOLED[v]["obs"]:.3f}')

t    : 146 sites, obs pooled = 9.375
pr   : 159 sites, obs pooled = 1384.816
wind :  16 sites, obs pooled = 2.258


In [14]:
# ---- pooled obs vs model (climatology means, network-level) ----
tab=[]
for v in ['t','pr','wind']:
    d=SITES[v]; p=POOLED[v]
    for m in MODELS:
        mm = d['obs_'+m].notna() & d[m].notna()
        ob=d.loc[mm,'obs_'+m]; mod=d.loc[mm,m]
        r2=np.corrcoef(ob,mod)[0,1]**2 if len(ob)>2 and np.std(ob)>0 and np.std(mod)>0 else np.nan
        tab.append(dict(var=v, model=MNAME[m], obs=ob.mean(), model_mean=mod.mean(),
                        bias=mod.mean()-ob.mean(),
                        rmse=np.sqrt(((mod-ob)**2).mean()), r2=r2, n=int(mm.sum())))
PTAB=pd.DataFrame(tab)
print(PTAB.to_string(index=False))

 var   model         obs  model_mean        bias        rmse       r2   n
   t      D3    9.375487    8.328339   -1.047148    1.562665 0.781560 146
   t     D15    9.375487    7.387238   -1.988249    2.580690 0.607310 146
   t     D75    9.375487    6.329220   -3.046267    3.574767 0.461872 146
   t CanESM2    9.375487    7.487378   -1.888109    2.675166 0.445785 146
   t CanRCM4    9.375487    5.876098   -3.499389    4.115138 0.568193 146
  pr      D3 1384.815912 1196.032010 -188.783902  366.956322 0.891942 159
  pr     D15 1384.815912 2280.761799  895.945887 1049.086861 0.855850 159
  pr     D75 1384.815912 2486.932832 1102.116920 1239.129567 0.690560 159
  pr CanESM2 1384.815912 1270.654631 -114.161281  839.278729 0.112361 159
  pr CanRCM4 1384.815912 1533.648038  148.832126  521.227045 0.689616 159
wind      D3    2.257538    3.055479    0.797942    1.042516 0.150125  16
wind     D15    2.257538    3.106836    0.849298    1.129972 0.182411  16
wind     D75    2.257538    2.635339  

In [15]:
# ---- per-station climatology + bias (T example shown; full in pickle) ----
tS=SITES['t'].copy()
tS['bias_D03']=tS['D03']-tS['obs_D03']
tS[['n_years','obs','D03','bias_D03','CanESM2','CanRCM4']].sort_values('obs',ascending=False).head(12).to_string(float_format=lambda x:'%.3f'%x)

'             n_years    obs    D03  bias_D03  CanESM2  CanRCM4\nsid                                                           \nUSC00356749       20 12.704 11.842    -0.862   10.613    9.952\nUSW00024229       20 12.640 11.425    -1.215   10.629    9.801\nUSC00352997       19 12.060 10.519    -1.541   10.696    9.156\nUSC00458278       16 11.935 11.370    -0.566    9.536    8.851\nUSC00451276       19 11.931  9.406    -2.525   10.219    8.810\nUSC00454169       15 11.865 11.645    -0.220    9.523    8.900\nUSC00454769       17 11.636 10.751    -0.885   10.574    8.608\nUSC00457727       19 11.629 10.805    -0.824    9.264    9.603\nUSC00459074       19 11.590 13.132     1.543    8.520    5.782\nUSC00451760       17 11.563  8.604    -2.960   10.499    6.687\nUSW00024233       20 11.549 10.999    -0.549    9.335    8.862\nUSC00458773       20 11.361 11.337    -0.023   10.651    9.742'

In [16]:
# ---- save pickle ----
OUT=os.path.join(BASE,'pickles','climate_land.pkl')
with open(OUT,'wb') as f:
    pickle.dump({'sites':SITES,'pooled':POOLED,'pooled_table':PTAB,
                 'n_sites':{v:len(SITES[v]) for v in SITES}}, f)
print('wrote', OUT)

wrote /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/pickles/climate_land.pkl


In [17]:
# ---- figure: obs climatology (top) + D03 bias (bottom) per variable, cartopy PlateCarree (offline: gridlines, no coastlines) ----
import cartopy.crs as ccrs
fig, axes = plt.subplots(2, 3, figsize=(15, 7), subplot_kw={'projection': ccrs.PlateCarree()})
LON = D03.set_index('sid')['lon']; LAT = D03.set_index('sid')['lat']
EXTENT = [-132, -116, 45, 54]
for j, (v, lbl) in enumerate(VARS.items()):
    d = SITES[v].copy()
    d['bias'] = d['D03'] - d['obs_D03']
    ok = d['obs'].notna(); dd = d[ok]
    sc = axes[0, j].scatter(LON[dd.index], LAT[dd.index], c=dd['obs'], cmap='RdBu_r', s=22, transform=ccrs.PlateCarree())
    axes[0, j].set_title(lbl + ' obs climatology (D03)')
    axes[0, j].gridlines(draw_labels=True, linewidth=0.3, color='grey', alpha=0.5, linestyle=':')
    axes[0, j].set_extent(EXTENT, crs=ccrs.PlateCarree()); axes[0, j].set_aspect('equal')
    fig.colorbar(sc, ax=axes[0, j], fraction=0.046, pad=0.04)
    okb = d['bias'].notna(); db = d[okb]
    val = db['bias'] * 100.0 / db['obs'] if v == 'wind' else db['bias']
    sc = axes[1, j].scatter(LON[db.index], LAT[db.index], c=val, cmap='RdBu_r', s=22, transform=ccrs.PlateCarree())
    axes[1, j].set_title(lbl + ' D03 ' + ('% bias' if v == 'wind' else 'bias'))
    axes[1, j].gridlines(draw_labels=True, linewidth=0.3, color='grey', alpha=0.5, linestyle=':')
    axes[1, j].set_extent(EXTENT, crs=ccrs.PlateCarree()); axes[1, j].set_aspect('equal')
    fig.colorbar(sc, ax=axes[1, j], fraction=0.046, pad=0.04)
fig.tight_layout()
OUTF = os.path.join(BASE, 'figures', 'climate', 'climate_land_map.png')
fig.savefig(OUTF, dpi=150, bbox_inches='tight'); print('saved', OUTF)
plt.close(fig)

saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/climate/climate_land_map.png


In [18]:
# (bias map merged into the 2-row map cell above)


## Interpretation

- Pooled obs vs model table above is the network-level *climate* comparison (period-mean climatology, not anomalies). 
- Per-site `n_years` shows which records are 20-yr vs sparse.
- Bias maps show where the D03 nest systematically over/under-states the regional climatology.

In [19]:
# ---- (b) emit a .tex snippet of the pooled obs-vs-model climatology table ----
import numpy as np
def cell_bias(x):
    if not np.isfinite(x): return 'NA'
    if x == 0: return '$0$'
    if x < 0:
        a = abs(x)
        return '$-%.2f$' % a
    return '$+%s$' % ('%.2f' % x)
def cell(x, nd=2):
    return 'NA' if not np.isfinite(x) else f'{x:.{nd}f}'
lines = []
lines.append('% Land climatology: pooled obs vs model (period means; bias = model - obs)')
lines.append(r'\begin{tabular}{l l r r r r}')
lines.append(r'\toprule')
lines.append('Var & Model & obs & bias & RMSE & R$^2$ \\ ')
lines.append(r'\midrule')
for _, r in PTAB.iterrows():
    lines.append(f"{r['var']:>4s} & {r['model']:>8s} & {cell(r['obs'])} & {cell_bias(r['bias'])} & {cell(r['rmse'])} & {cell(r['r2'])} \\ ")
lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
tex = '\n'.join(lines)
print(tex)
with open(os.path.join(BASE, 'report', 'climate_land_table.tex'), 'w') as f: f.write(tex + '\n')
print('wrote report/climate_land_table.tex')


% Land climatology: pooled obs vs model (period means; bias = model - obs)
\begin{tabular}{l l r r r r}
\toprule
Var & Model & obs & bias & RMSE & R$^2$ \ 
\midrule
   t &       D3 & 9.38 & $-1.05$ & 1.56 & 0.78 \ 
   t &      D15 & 9.38 & $-1.99$ & 2.58 & 0.61 \ 
   t &      D75 & 9.38 & $-3.05$ & 3.57 & 0.46 \ 
   t &  CanESM2 & 9.38 & $-1.89$ & 2.68 & 0.45 \ 
   t &  CanRCM4 & 9.38 & $-3.50$ & 4.12 & 0.57 \ 
  pr &       D3 & 1384.82 & $-188.78$ & 366.96 & 0.89 \ 
  pr &      D15 & 1384.82 & $+895.95$ & 1049.09 & 0.86 \ 
  pr &      D75 & 1384.82 & $+1102.12$ & 1239.13 & 0.69 \ 
  pr &  CanESM2 & 1384.82 & $-114.16$ & 839.28 & 0.11 \ 
  pr &  CanRCM4 & 1384.82 & $+148.83$ & 521.23 & 0.69 \ 
wind &       D3 & 2.26 & $+0.80$ & 1.04 & 0.15 \ 
wind &      D15 & 2.26 & $+0.85$ & 1.13 & 0.18 \ 
wind &      D75 & 2.26 & $+0.38$ & 0.99 & 0.00 \ 
wind &  CanESM2 & 2.26 & $+1.72$ & 2.08 & 0.01 \ 
wind &  CanRCM4 & 2.26 & $+1.53$ & 1.84 & 0.09 \ 
\bottomrule
\end{tabular}
wrote report/climate_

In [20]:
# ---- (c) land P rescale: mm/yr -> mm/day (report Table 1 units) ----
PTAB['obs_mmday']  = [ (r['obs']/365.25)  if (r['var']=='pr' and np.isfinite(r['obs']))  else np.nan for _, r in PTAB.iterrows() ]
PTAB['rmse_mmday'] = [ (r['rmse']/365.25) if (r['var']=='pr' and np.isfinite(r['rmse'])) else np.nan for _, r in PTAB.iterrows() ]
print(PTAB.to_string(index=False, float_format=lambda x:'%.3f'%x))
with open(os.path.join(BASE,'pickles','climate_land.pkl'),'rb') as f: CL = pickle.load(f)
CL['pr_obs_mmday']  = PTAB['obs_mmday'].tolist()
CL['pr_rmse_mmday'] = PTAB['rmse_mmday'].tolist()
with open(os.path.join(BASE,'pickles','climate_land.pkl'),'wb') as f: pickle.dump(CL, f)
print('pickles/climate_land.pkl updated (pr_obs_mmday / pr_rmse_mmday appended)')


 var   model      obs  model_mean     bias     rmse    r2   n  obs_mmday  rmse_mmday
   t      D3    9.375       8.328   -1.047    1.563 0.782 146        NaN         NaN
   t     D15    9.375       7.387   -1.988    2.581 0.607 146        NaN         NaN
   t     D75    9.375       6.329   -3.046    3.575 0.462 146        NaN         NaN
   t CanESM2    9.375       7.487   -1.888    2.675 0.446 146        NaN         NaN
   t CanRCM4    9.375       5.876   -3.499    4.115 0.568 146        NaN         NaN
  pr      D3 1384.816    1196.032 -188.784  366.956 0.892 159      3.791       1.005
  pr     D15 1384.816    2280.762  895.946 1049.087 0.856 159      3.791       2.872
  pr     D75 1384.816    2486.933 1102.117 1239.130 0.691 159      3.791       3.393
  pr CanESM2 1384.816    1270.655 -114.161  839.279 0.112 159      3.791       2.298
  pr CanRCM4 1384.816    1533.648  148.832  521.227 0.690 159      3.791       1.427
wind      D3    2.258       3.055    0.798    1.043 0.150  16    